# ifcfill label encoding for synthetic data

This notebook shows `cat_encoding="label"`, which converts categorical variables into integer codes and stores mappings so `inverse_transform` can decode them later.

## Setup

Install ifcfill with the optional notebook dependencies:

```bash
pip install "ifcfill[examples]"
```

In [1]:
import pandas as pd

from ifcfill import IFCTransformer

## Create sample data

In [2]:
df = pd.DataFrame(
    {
        "age": [25, 30, None, 40, 35],
        "salary": [50_000.50, None, 75_000.00, 90_000.25, 62_000.00],
        "city": ["London", None, "Paris", "London", "Amman"],
        "zip_code": ["00123", "00456", None, "00123", "07890"],
        "joined": pd.to_datetime(
            ["2020-01-01", "2021-06-15", None, "2023-03-10", "2022-11-01"]
        ),
        "active": ["yes", "yes", "yes", "yes", "yes"],
    }
)

df

,age,salary,city,zip_code,joined,active
0,25.0,50000.50,London,00123,2020-01-01,yes
1,30.0,NaN,NaN,00456,2021-06-15,yes
2,NaN,75000.00,Paris,NaN,NaT,yes
3,40.0,90000.25,London,00123,2023-03-10,yes
4,35.0,62000.00,Amman,07890,2022-11-01,yes


## Fit and transform with label encoding

Encodings are unsupervised and invertible, so transformed data can be used by synthetic data generators and later mapped back to the original table structure.

In [3]:
transformer = IFCTransformer(
    col_types={"zip_code": "categorical"},
    cat_fill="constant",
    cat_constant="UNKNOWN",
    cat_encoding="label",
)

transformed = transformer.fit_transform(df)
transformed

,age,salary,city,zip_code,joined
0,25,50000.5000,1,0,18262
1,30,69250.1875,3,1,18793
2,32,75000.0000,2,3,19045
3,40,90000.2500,1,0,19426
4,35,62000.0000,0,2,19297


`city` and `zip_code` are now integer-coded categorical variables.

In [4]:
transformed.dtypes

age           int64
salary      float64
city          int64
zip_code      int64
joined        int64
dtype: object

## Inspect the category mappings

In [5]:
transformer.category_mappings_

{'city': {'Amman': 0, 'London': 1, 'Paris': 2, 'UNKNOWN': 3},
 'zip_code': {'00123': 0, '00456': 1, '07890': 2, 'UNKNOWN': 3}}

In [6]:
transformer.inverse_category_mappings_

{'city': {0: 'Amman', 1: 'London', 2: 'Paris', 3: 'UNKNOWN'},
 'zip_code': {0: '00123', 1: '00456', 2: '07890', 3: 'UNKNOWN'}}

## Inverse transform generated-like data

Synthetic generators may return float values for encoded categorical columns. `inverse_transform` rounds and clips those values to the known code range before decoding. If the decoded value is the learned missing category, it becomes a missing value again.

In [7]:
generated_like = transformed.copy()
generated_like["city"] = [0.1, 1.8, 20.0, -3.0, 1.2]

restored = transformer.inverse_transform(generated_like)
restored

,age,salary,city,zip_code,joined,active
0,25,50000.5000,Amman,00123,18262,yes
1,30,69250.1875,Paris,00456,18793,yes
2,32,75000.0000,NaN,NaN,19045,yes
3,40,90000.2500,Amman,00123,19426,yes
4,35,62000.0000,London,07890,19297,yes
